# 10 · DREF Sankey Diagrams

This notebook builds two presentation-ready Plotly Sankey diagrams directly from the master workbook.

- Figure 1: Pillar -> Appeal Type -> Top 20 Countries by CHF
- Figure 2: Pillar -> Appeal Type -> Hazard Type by CHF

Scope:
- Use the `Year` column to keep all available 2026 rows.
- Do not apply a Q1 cutoff.
- Drop only unusable rows with missing approval dates or missing/zero CHF totals.
- Export high-resolution PNGs to `outputs/implementation_phase_python`.
- Display both figures inline for review in the notebook.

## 1. Load Workbook and Inspect Required Columns

Load the workbook directly from the repo root, inspect the source columns we need, and confirm that the `Year`, approval date, CHF total, country, appeal type, pillar, and hazard fields are available.

In [13]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings("ignore", category=DeprecationWarning)
pio.templates.default = "plotly_white"

ROOT = Path("..").resolve()
WORKBOOK_PATH = ROOT / "DREF_MasterDataset_v1.1 .xlsx"
OUTPUT_DIR = ROOT / "outputs" / "implementation_phase_python"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SHEET_CANDIDATES = ["ALL_DATA", "ALL_Data"]
YEAR_TARGET = 2026

COUNTRY_EXCLUSIONS = {
    "Africa Secretariat",
    "America Secretariat",
    "MENA Secretariat",
}

APPEAL_FAMILY_MAP = {
    "DREF": "DREF",
    "i-DREF": "DREF",
    "a-DREF": "DREF",
    "EA": "EA",
    "EAP": "EAP",
    "s-EAP": "EAP",
}

PILLAR_ORDER = ["Response", "Anticipatory"]
PILLAR_COLORS = {
    "Response": "#2C7178",
    "Anticipatory": "#AD914C",
    "Other": "#8C8C8C",
}
APPEAL_COLORS = {
    "DREF": "#B4612D",
    "EA": "#345C8C",
    "EAP": "#627A54",
}


def format_chf(value: float) -> str:
    return f"CHF {value:,.0f}"


def clean_text(value, default="Unknown"):
    if pd.isna(value):
        return default
    text = str(value).strip()
    return text if text and text.lower() != "nan" else default


def normalize_pillar(value) -> str:
    text = clean_text(value, default="Other").lower()
    if "anticip" in text:
        return "Anticipatory"
    if "response" in text:
        return "Response"
    return "Other"


def normalize_appeal(value) -> str:
    text = clean_text(value, default="")
    return APPEAL_FAMILY_MAP.get(text, "Unknown")


def load_master_workbook() -> tuple[pd.DataFrame, str]:
    for sheet_name in SHEET_CANDIDATES:
        try:
            frame = pd.read_excel(WORKBOOK_PATH, sheet_name=sheet_name)
            return frame, sheet_name
        except ValueError:
            continue
    raise ValueError(f"Could not find one of the expected sheets: {SHEET_CANDIDATES}")


def ensure_year_column(frame: pd.DataFrame) -> pd.Series:
    year_columns = [column for column in frame.columns if str(column).strip().lower() == "year"]
    if not year_columns:
        raise KeyError("The workbook does not contain a Year column.")
    year_series = pd.to_numeric(frame[year_columns[0]], errors="coerce")
    return year_series

In [14]:
raw_df, sheet_name = load_master_workbook()
raw_df = raw_df.rename(columns=lambda column: str(column).strip())
raw_df["year"] = ensure_year_column(raw_df)

approval_date_column = "Date of Approval EnC (start date)"
chf_column = "Total Approved (CHF)"
required_columns = ["Country", "Appeal Type", "Pillar", "Disaster Definition", approval_date_column, chf_column, "year"]
existing_columns = [column for column in required_columns if column in raw_df.columns or column == "year"]

print(f"Workbook: {WORKBOOK_PATH.name}")
print(f"Sheet   : {sheet_name}")
print(f"Rows    : {len(raw_df):,}")
print(f"Columns : {len(raw_df.columns):,}")
print("Required columns found:")
for column in existing_columns:
    print(f"- {column}")

missing_columns = [column for column in required_columns if column not in existing_columns]
if missing_columns:
    print("Missing columns:", missing_columns)

raw_df.head(3)

Workbook: DREF_MasterDataset_v1.1 .xlsx
Sheet   : ALL_DATA
Rows    : 2,506
Columns : 79
Required columns found:
- Country
- Appeal Type
- Pillar
- Disaster Definition
- Date of Approval EnC (start date)
- Total Approved (CHF)
- year


c:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Index,Appeal ID,Code Number,Child ID,Appeal's allocation,Pillar,Appeal Type,Allocation Type,Country,Region Code,...,Canada Gvt,ECHO,NLRC,Belgian gvt,Other Donor (CHF),Donor / PNS,Reimbursed (For Finance),Comments,Year,year
0,1,MDRCD006,006,NaN,NaN,Anticipatory,i-DREF,Grant,Democratic Republic of the Congo,AF,...,NaN,NaN,67659.0,NaN,NaN,NaN,1083.0,NaN,2009,2009
1,2,MDRTG002,002,NaN,NaN,Response,DREF,Grant,Togo,AF,...,NaN,NaN,14961.0,NaN,NaN,NaN,4257.0,NaN,2009,2009
2,3,MDRCF003,003,NaN,NaN,Response,DREF,Grant,Central African Republic,AF,...,NaN,NaN,14852.0,NaN,NaN,NaN,1638.0,NaN,2009,2009


## 2. Filter 2026 Records Using the Year Column

Keep every available 2026 record in the workbook by filtering on the `Year` column. This notebook does not apply a Q1 cutoff.

In [15]:
df_2026 = raw_df.loc[raw_df["year"].eq(YEAR_TARGET)].copy()
df_2026[approval_date_column] = pd.to_datetime(df_2026[approval_date_column], errors="coerce")
df_2026[chf_column] = pd.to_numeric(df_2026[chf_column], errors="coerce")

df_2026 = df_2026.loc[df_2026[approval_date_column].notna()].copy()

print(f"2026 rows before unusable-row cleanup: {len(df_2026):,}")
print(f"2026 CHF total before cleanup      : {format_chf(df_2026[chf_column].sum())}")

df_2026.head(3)

2026 rows before unusable-row cleanup: 73
2026 CHF total before cleanup      : CHF 26,036,816


,Index,Appeal ID,Code Number,Child ID,Appeal's allocation,Pillar,Appeal Type,Allocation Type,Country,Region Code,...,Canada Gvt,ECHO,NLRC,Belgian gvt,Other Donor (CHF),Donor / PNS,Reimbursed (For Finance),Comments,Year,year
2433,2439,MDRGM017,017,NaN,First,Response,DREF,Grant,Gambia,AF,...,NaN,125000.0,NaN,NaN,NaN,NaN,NaN,NaN,2026,2026
2434,2440,MDRCO034,034,NaN,First,Anticipatory,i-DREF,Grant,Colombia,AM,...,NaN,50000.0,NaN,NaN,NaN,NaN,NaN,NaN,2026,2026
2435,2441,MDRMZ021,021,NaN,Second,Anticipatory,EAP,Grant,Mozambique,AF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,2026


## 3. Remove Unusable Rows

Drop only unusable records with missing or zero CHF totals, while keeping all allocation types in scope.

In [16]:
df_2026 = df_2026.loc[df_2026[chf_column].notna() & df_2026[chf_column].gt(0)].copy()

print(f"2026 rows after cleanup            : {len(df_2026):,}")
print(f"2026 CHF total after cleanup       : {format_chf(df_2026[chf_column].sum())}")
print(f"Unique allocation types in 2026 set : {df_2026.get('Allocation Type', pd.Series(dtype=str)).nunique(dropna=True)}")

2026 rows after cleanup            : 73
2026 CHF total after cleanup       : CHF 26,036,816
Unique allocation types in 2026 set : 2


## 4. Prepare Data for Plotting

Clean the key fields, derive display categories, and build the aggregated link tables for each Sankey diagram.

In [17]:
base = df_2026.copy()
base["country_clean"] = base["Country"].map(clean_text)
base["pillar_clean"] = base["Pillar"].map(normalize_pillar)
base["appeal_family"] = base["Appeal Type"].map(normalize_appeal)
base["hazard_clean"] = base["Disaster Definition"].map(lambda value: clean_text(value, default="Unknown"))
base["allocation_clean"] = base.get("Allocation Type", pd.Series(index=base.index, dtype=object)).map(clean_text)

country_scope = base.loc[~base["country_clean"].isin(COUNTRY_EXCLUSIONS) & base["country_clean"].ne("Unknown")].copy()
country_rank = (
    country_scope.groupby("country_clean", as_index=False)[chf_column]
    .sum()
    .sort_values(chf_column, ascending=False)
)
country_top20 = country_rank.head(20)["country_clean"].tolist()
country_scope["country_bucket"] = np.where(
    country_scope["country_clean"].isin(country_top20),
    country_scope["country_clean"],
    "Other Countries",
)

hazard_scope = base.copy()
hazard_rank = (
    hazard_scope.groupby("hazard_clean", as_index=False)[chf_column]
    .sum()
    .sort_values(chf_column, ascending=False)
)
hazard_top = hazard_rank.head(12)["hazard_clean"].tolist()
hazard_scope["hazard_bucket"] = np.where(
    hazard_scope["hazard_clean"].isin(hazard_top),
    hazard_scope["hazard_clean"],
    "Other Hazards",
)

country_stage_1 = (
    country_scope.groupby(["pillar_clean", "appeal_family"], as_index=False)[chf_column]
    .sum()
)
country_stage_2 = (
    country_scope.groupby(["appeal_family", "country_bucket"], as_index=False)[chf_column]
    .sum()
)
country_terminal_order = country_top20 + (["Other Countries"] if "Other Countries" in country_scope["country_bucket"].unique() else [])

hazard_stage_1 = (
    hazard_scope.groupby(["pillar_clean", "appeal_family"], as_index=False)[chf_column]
    .sum()
)
hazard_stage_2 = (
    hazard_scope.groupby(["appeal_family", "hazard_bucket"], as_index=False)[chf_column]
    .sum()
)
hazard_terminal_order = hazard_top + (["Other Hazards"] if "Other Hazards" in hazard_scope["hazard_bucket"].unique() else [])

print("Country ranking preview:")
display(country_rank.head(10).style.format({chf_column: format_chf}))
print("Hazard ranking preview:")
display(hazard_rank.head(10).style.format({chf_column: format_chf}))

Country ranking preview:


,country_clean,Total Approved (CHF)
29,Lebanon,"CHF 2,492,946"
11,Cameroon,"CHF 2,299,782"
25,Iran,"CHF 1,525,304"
0,Afghanistan,"CHF 1,500,000"
31,Madagascar,"CHF 1,394,995"
35,Mozambique,"CHF 1,148,816"
10,Burkina Faso,"CHF 876,576"
27,Kenya,"CHF 718,144"
34,Morocco,"CHF 637,064"
50,Venezuela,"CHF 591,383"


Hazard ranking preview:


,hazard_clean,Total Approved (CHF)
6,Flood,"CHF 10,037,730"
1,Complex Emergency,"CHF 4,750,299"
4,Epidemic,"CHF 2,327,023"
2,Cyclone,"CHF 1,618,717"
11,Population Movement,"CHF 1,388,247"
5,Fire,"CHF 1,350,665"
9,Other,"CHF 1,262,013"
10,Pluvial/Flash Flood,"CHF 903,217"
7,Food Insecurity,"CHF 849,785"
12,Storm Surge,"CHF 649,892"


## 5. Create the Two Figures

Build a reusable Sankey helper, then generate the country and hazard figures with large labels, stage headers, and a polished presentation layout.

In [22]:
def format_compact(value: float) -> str:
    """Format a CHF value as compact string: 2.2M, 300K, etc."""
    if value >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"
    elif value >= 1_000:
        return f"{value / 1_000:.0f}K"
    else:
        return f"{value:.0f}"


def make_node_label(text: str, amount: float, width: int = 18) -> str:
    """Bold-wrapped label with CHF amount inline on the same last line as the name.
    Example: 'Flood   CHF 10.0M'  or  'Population Movement   CHF 1.4M'
    """
    words = str(text).split()
    if not words:
        return text
    lines, line = [], words[0]
    for word in words[1:]:
        if len(line) + 1 + len(word) <= width:
            line = f"{line} {word}"
        else:
            lines.append(line)
            line = word
    lines.append(line)
    # Append the amount inline on the last line, separated by 3 spaces
    lines[-1] = f"{lines[-1]}   CHF {format_compact(amount)}"
    return "<b>" + "<br>".join(lines) + "</b>"


def rgba(hex_color: str, alpha: float) -> str:
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r}, {g}, {b}, {alpha})"


def build_node_positions(stage_values: list[list[str]]) -> tuple[list[float], list[float]]:
    x_positions, y_positions = [], []
    stage_x = [0.02, 0.50, 0.98]
    for stage_index, labels in enumerate(stage_values):
        count = max(len(labels), 1)
        y_values = [0.5] if count == 1 else np.linspace(0.05, 0.95, count).tolist()
        x_positions.extend([stage_x[stage_index]] * count)
        y_positions.extend(y_values)
    return x_positions, y_positions


def build_three_stage_sankey(
    stage_1: pd.DataFrame,
    stage_2: pd.DataFrame,
    terminal: pd.DataFrame,
    terminal_labels: list[str],
    terminal_title: str,
    title: str,
    subtitle: str,
    output_stem: str,
) -> go.Figure:
    stage_1_nodes = [lbl for lbl in PILLAR_ORDER if lbl in stage_1["pillar_clean"].unique()]
    stage_2_nodes = [lbl for lbl in ["DREF", "EA", "EAP"]
                     if lbl in pd.concat([stage_1["appeal_family"], stage_2["appeal_family"]]).unique()]
    stage_3_nodes = terminal_labels

    # ── CHF totals per node for inline labels ─────────────────────────────────
    pillar_totals = stage_1.groupby("pillar_clean")[chf_column].sum()
    appeal_totals = stage_2.groupby("appeal_family")[chf_column].sum()
    terminal_col  = stage_2.columns[1]
    term_totals   = stage_2.groupby(terminal_col)[chf_column].sum()

    node_labels = (
        [make_node_label(lbl, pillar_totals.get(lbl, 0)) for lbl in stage_1_nodes]
      + [make_node_label(lbl, appeal_totals.get(lbl, 0)) for lbl in stage_2_nodes]
      + [make_node_label(lbl, term_totals.get(lbl, 0))   for lbl in stage_3_nodes]
    )
    node_x, node_y = build_node_positions([stage_1_nodes, stage_2_nodes, stage_3_nodes])

    node_colors = [PILLAR_COLORS.get(lbl, PILLAR_COLORS["Other"]) for lbl in stage_1_nodes]
    node_colors.extend(APPEAL_COLORS.get(lbl, "#8D99AE") for lbl in stage_2_nodes)
    palette = px.colors.qualitative.Set3 + px.colors.qualitative.Pastel + px.colors.qualitative.Vivid
    node_colors.extend(palette[i % len(palette)] for i in range(len(stage_3_nodes)))

    node_index = {lbl: idx for idx, lbl in enumerate(stage_1_nodes + stage_2_nodes + stage_3_nodes)}

    s1 = stage_1.copy()
    s1["source"] = s1["pillar_clean"].map(node_index)
    s1["target"] = s1["appeal_family"].map(node_index)
    s1["color"]  = s1["pillar_clean"].map(lambda v: rgba(PILLAR_COLORS.get(v, PILLAR_COLORS["Other"]), 0.38))

    s2 = stage_2.copy()
    s2["source"] = s2["appeal_family"].map(node_index)
    s2["target"] = s2.iloc[:, 1].map(node_index)
    s2["color"]  = s2["appeal_family"].map(lambda v: rgba(APPEAL_COLORS.get(v, "#8D99AE"), 0.32))

    links = pd.concat([s1, s2], ignore_index=True)

    term_qa = (terminal.groupby(terminal_col, as_index=False)[chf_column]
               .sum().sort_values(chf_column, ascending=False))
    print(f"{terminal_title} terminal nodes: {len(stage_3_nodes):,}")
    display(term_qa.head(14).style.format({chf_column: format_chf}))

    # ── Canvas & margin ───────────────────────────────────────────────────────
    CANVAS_H = 1220
    MARGIN_T = 320
    MARGIN_B = 60
    PLOT_H   = CANVAS_H - MARGIN_T - MARGIN_B  # 840 px

    TITLE_Y     = 1.0 + 300 / PLOT_H
    SUBTITLE_Y  = 1.0 + 220 / PLOT_H
    STAGE_HDR_Y = 1.0 +  40 / PLOT_H

    fig = go.Figure(
        go.Sankey(
            arrangement="fixed",
            node=dict(
                pad=22,
                thickness=22,
                line=dict(color="rgba(60, 60, 60, 0.35)", width=0.8),
                label=node_labels,
                color=node_colors,
                x=node_x,
                y=node_y,
                hovertemplate="%{label}<extra></extra>",
            ),
            link=dict(
                source=links["source"],
                target=links["target"],
                value=links[chf_column],
                color=links["color"],
                hovertemplate="CHF %{value:,.0f}<extra></extra>",
            ),
        )
    )

    fig.update_layout(
        title=dict(text=""),
        annotations=[
            dict(
                text=f"<b>{title}</b>",
                x=0.5, y=TITLE_Y,
                xref="paper", yref="paper",
                showarrow=False,
                xanchor="center", yanchor="top",
                font=dict(size=34, family="Arial", color="#1a1a1a"),
            ),
            dict(
                text=subtitle,
                x=0.5, y=SUBTITLE_Y,
                xref="paper", yref="paper",
                showarrow=False,
                xanchor="center", yanchor="top",
                font=dict(size=18, color="#4a4a4a", family="Arial"),
                align="center",
            ),
            dict(
                text="<b>Pillar</b>",
                x=0.02, y=STAGE_HDR_Y,
                xref="paper", yref="paper",
                showarrow=False,
                xanchor="left", yanchor="bottom",
                font=dict(size=24, family="Arial"),
            ),
            dict(
                text="<b>Appeal Type</b>",
                x=0.50, y=STAGE_HDR_Y,
                xref="paper", yref="paper",
                showarrow=False,
                xanchor="center", yanchor="bottom",
                font=dict(size=24, family="Arial"),
            ),
            dict(
                text=f"<b>{terminal_title}</b>",
                x=0.98, y=STAGE_HDR_Y,
                xref="paper", yref="paper",
                showarrow=False,
                xanchor="right", yanchor="bottom",
                font=dict(size=24, family="Arial"),
            ),
        ],
        font=dict(size=20, family="Arial"),
        width=1800,
        height=CANVAS_H,
        margin=dict(l=70, r=70, t=MARGIN_T, b=MARGIN_B),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )

    output_path = OUTPUT_DIR / f"{output_stem}.png"
    fig.write_image(str(output_path), format="png", width=1800, height=CANVAS_H, scale=3)
    print(f"Saved PNG -> {output_path}")
    return fig

## 6. Display Figures Inline

Render both Sankey diagrams directly in the notebook for immediate review.

In [23]:
country_title = "DREF 2026 Funds by Top 20 Countries"
country_subtitle = "2026 approvals by Top 20 Countries"
country_fig = build_three_stage_sankey(
    country_stage_1,
    country_stage_2,
    country_stage_2,
    country_terminal_order,
    "Country",
    country_title,
    country_subtitle,
    "dref_sankey_top20_countries_2026",
)

hazard_title = "DREF 2026 Funds by Hazard Type"
hazard_subtitle = "2026 approvals by Hazard Categories"
hazard_fig = build_three_stage_sankey(
    hazard_stage_1,
    hazard_stage_2,
    hazard_stage_2,
    hazard_terminal_order,
    "Hazard type",
    hazard_title,
    hazard_subtitle,
    "dref_sankey_hazard_type_2026",
)

country_fig.show()
hazard_fig.show()

Country terminal nodes: 21


,country_bucket,Total Approved (CHF)
15,Other Countries,"CHF 7,861,169"
10,Lebanon,"CHF 2,492,946"
4,Cameroon,"CHF 2,299,782"
8,Iran,"CHF 1,525,304"
0,Afghanistan,"CHF 1,500,000"
12,Madagascar,"CHF 1,394,995"
14,Mozambique,"CHF 1,148,816"
3,Burkina Faso,"CHF 876,576"
9,Kenya,"CHF 718,144"
13,Morocco,"CHF 637,064"


Saved PNG -> C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\outputs\implementation_phase_python\dref_sankey_top20_countries_2026.png
Hazard type terminal nodes: 13


,hazard_bucket,Total Approved (CHF)
5,Flood,"CHF 10,037,730"
1,Complex Emergency,"CHF 4,750,299"
3,Epidemic,"CHF 2,327,023"
2,Cyclone,"CHF 1,618,717"
11,Population Movement,"CHF 1,388,247"
4,Fire,"CHF 1,350,665"
8,Other,"CHF 1,262,013"
10,Pluvial/Flash Flood,"CHF 903,217"
6,Food Insecurity,"CHF 849,785"
12,Storm Surge,"CHF 649,892"


Saved PNG -> C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\outputs\implementation_phase_python\dref_sankey_hazard_type_2026.png


## 7. Export High-Resolution PNGs

The figures are exported as high-resolution PNG files during figure creation. This section confirms the saved outputs.

In [20]:
country_png = OUTPUT_DIR / "dref_sankey_top20_countries_2026.png"
hazard_png = OUTPUT_DIR / "dref_sankey_hazard_type_2026.png"

print(f"Country PNG exists: {country_png.exists()} -> {country_png}")
print(f"Hazard PNG exists : {hazard_png.exists()} -> {hazard_png}")

Country PNG exists: True -> C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\outputs\implementation_phase_python\dref_sankey_top20_countries_2026.png
Hazard PNG exists : True -> C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\outputs\implementation_phase_python\dref_sankey_hazard_type_2026.png


## 8. Optional Static Export Cells for Slide-Ready Output

If you want larger slide assets later, switch on the toggle below and rerun the cell.

In [21]:
EXPORT_SLIDE_READY = False

if EXPORT_SLIDE_READY:
    country_slide_path = OUTPUT_DIR / "dref_sankey_top20_countries_2026_slide.png"
    hazard_slide_path = OUTPUT_DIR / "dref_sankey_hazard_type_2026_slide.png"
    country_fig.write_image(str(country_slide_path), format="png", width=2400, height=1500, scale=3)
    hazard_fig.write_image(str(hazard_slide_path), format="png", width=2400, height=1500, scale=3)
    print(f"Saved slide-ready PNGs to {country_slide_path} and {hazard_slide_path}")
else:
    print("Slide-ready export is disabled. Set EXPORT_SLIDE_READY = True to generate the larger PNG versions.")

Slide-ready export is disabled. Set EXPORT_SLIDE_READY = True to generate the larger PNG versions.
